# LeetCode 2085 — Count Common Words With One Occurrence bằng Python

Link đề: https://leetcode.com/problems/count-common-words-with-one-occurrence/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, String, Counting

```python
class Solution:
    def countWords(self, words1: List[str], words2: List[str]) -> int:

```

## 1. Mục tiêu học

Sau bài này bạn sẽ:

1. Hiểu đề: đếm các chuỗi xuất hiện **đúng 1 lần ở mảng thứ nhất** và **đúng 1 lần ở mảng thứ hai**.
2. Biết dùng `dict`, `set` và `collections.Counter` để **đếm tần suất** — kỹ thuật nền tảng của rất nhiều bài Easy/Medium.
3. So sánh cách brute force dùng `list.count` (chậm) với cách dùng hash table (nhanh).
4. Học thêm một thủ thuật: gom hai mảng vào **một** dictionary với "trạng thái" cho từng từ.
5. Tự trace được từng bước, viết test đối chiếu nhiều cách giải, và tránh các lỗi phổ biến (đếm sai, quên điều kiện "đúng 1 lần").

## 2. Đề bài & ví dụ

Cho hai mảng chuỗi `words1` và `words2`. Hãy trả về **số lượng chuỗi xuất hiện đúng một lần trong *cả hai* mảng**.

Nói cách khác, một chuỗi `w` được tính khi:

- `w` xuất hiện **đúng 1 lần** trong `words1`, và
- `w` xuất hiện **đúng 1 lần** trong `words2`.

| Ví dụ | words1 | words2 | Output |
|---|---|---|---|
| 1 | `["leetcode","is","amazing","as","is"]` | `["amazing","leetcode","is"]` | `2` |
| 2 | `["b","bb","bbb"]` | `["a","aa","aaa"]` | `0` |
| 3 | `["a","ab"]` | `["a","a","a","ab"]` | `1` |

Giải thích ví dụ 1: `"leetcode"` 1 lần ở mỗi mảng (tính); `"amazing"` 1 lần ở mỗi mảng (tính); `"is"` xuất hiện 2 lần ở `words1` (không tính); `"as"` chỉ có ở `words1` (không tính).

Constraints: `1 <= words1.length, words2.length <= 1000`, `1 <= words1[i].length, words2[j].length <= 30`, chỉ gồm chữ cái thường.

In [1]:
words1 = ["leetcode", "is", "amazing", "as", "is"]
words2 = ["amazing", "leetcode", "is"]
print("words1 =", words1)
print("words2 =", words2)
print("Số lần 'is' ở words1:", words1.count("is"), "| ở words2:", words2.count("is"))
print("Số lần 'amazing' ở words1:", words1.count("amazing"), "| ở words2:", words2.count("amazing"))
print("Kỳ vọng ví dụ 1 = 2")

words1 = ['leetcode', 'is', 'amazing', 'as', 'is']
words2 = ['amazing', 'leetcode', 'is']
Số lần 'is' ở words1: 2 | ở words2: 1
Số lần 'amazing' ở words1: 1 | ở words2: 1
Kỳ vọng ví dụ 1 = 2


## 3. Kiến thức nền

### 3.1. `list.count(x)` — đếm phần tử bằng cách duyệt hết list

`lst.count(x)` đi qua **toàn bộ** list và đếm số phần tử bằng `x`. Mỗi lần gọi tốn `O(n)`.

In [2]:
a = ["a", "b", "a", "c", "a"]
print(a.count("a"), a.count("b"), a.count("z"))   # 3 1 0

3 1 0


### 3.2. `dict` — bảng băm: tra cứu và cập nhật nhanh

`dict` lưu cặp `khóa -> giá trị`. Tra cứu / gán `d[key]` trung bình `O(1)`.

Để **đếm**, ta cộng dồn: `d[w] = d.get(w, 0) + 1`. `d.get(w, 0)` trả `0` nếu `w` chưa có, tránh lỗi `KeyError`.

In [3]:
d = {}
for w in ["x", "y", "x"]:
    d[w] = d.get(w, 0) + 1
print(d)
print(d.get("zzz", 0))     # 0, không báo lỗi
print("x" in d, "zzz" in d)

{'x': 2, 'y': 1}
0
True False


### 3.3. `collections.Counter` — dict chuyên để đếm

`Counter(iterable)` tạo sẵn bảng đếm. Truy cập khóa **không tồn tại** trả `0` (không lỗi) — rất tiện cho bài này.

In [4]:
from collections import Counter

c = Counter(["is", "leetcode", "is"])
print(c)
print(c["is"], c["leetcode"], c["absent"])     # 2 1 0
print([w for w, k in c.items() if k == 1])     # những từ xuất hiện đúng 1 lần

Counter({'is': 2, 'leetcode': 1})
2 1 0
['leetcode']


### 3.4. `set` — tập hợp và phép giao `&`

`set` không có phần tử trùng, kiểm tra `x in s` là `O(1)`. Phép giao `A & B` cho các phần tử **có ở cả hai** tập.

In [5]:
A = {"a", "b", "c"}
B = {"b", "c", "d"}
print(A & B)          # {'b', 'c'}
print(len(A & B))     # 2

{'b', 'c'}
2


## 4. Ý tưởng nền tảng — tính bằng tay

Ví dụ 1:

- `words1`: leetcode(1), is(2), amazing(1), as(1)  -> các từ "đúng 1 lần": **leetcode, amazing, as**
- `words2`: amazing(1), leetcode(1), is(1)         -> các từ "đúng 1 lần": **amazing, leetcode, is**

Một từ được tính khi nó nằm trong **cả hai** danh sách "đúng 1 lần": giao của `{leetcode, amazing, as}` và `{amazing, leetcode, is}` là `{leetcode, amazing}` -> đáp án `2`.

Hai bước tư duy: **(1)** với mỗi mảng, tìm các từ xuất hiện đúng 1 lần; **(2)** lấy giao và đếm.

In [6]:
once1 = {w for w in set(words1) if words1.count(w) == 1}
once2 = {w for w in set(words2) if words2.count(w) == 1}
print("once1 =", once1)
print("once2 =", once2)
print("giao  =", once1 & once2, "->", len(once1 & once2))

once1 = {'leetcode', 'as', 'amazing'}
once2 = {'is', 'amazing', 'leetcode'}
giao  = {'amazing', 'leetcode'} -> 2


## 5. Cách 1 — Brute Force: dùng `list.count`

Với mỗi từ **khác nhau** `w` của `words1`, gọi `words1.count(w)` và `words2.count(w)`; nếu cả hai bằng 1 thì cộng 1.

Ta duyệt `set(words1)` để mỗi từ chỉ xét một lần (nếu duyệt trực tiếp `words1` thì từ trùng bị xét nhiều lần; nhưng từ trùng thì `count != 1` nên cũng không bị đếm sai).

In [7]:
def count_words_brute(words1, words2):
    ans = 0
    for w in set(words1):
        if words1.count(w) == 1 and words2.count(w) == 1:
            ans += 1
    return ans

print(count_words_brute(["leetcode", "is", "amazing", "as", "is"], ["amazing", "leetcode", "is"]))  # 2
print(count_words_brute(["b", "bb", "bbb"], ["a", "aa", "aaa"]))                                    # 0
print(count_words_brute(["a", "ab"], ["a", "a", "a", "ab"]))                                        # 1

2
0
1


**Độ phức tạp:** gọi `n = len(words1)`, `m = len(words2)`. Có tới `n` từ khác nhau, mỗi từ gọi `count` tốn `O(n + m)` (và so sánh chuỗi tối đa 30 ký tự). Tổng `O(n·(n+m)·L)` với `L ≤ 30`; với `n, m ≤ 1000` vẫn chạy được, nhưng là **nút thắt**: ta đếm lại cùng một mảng rất nhiều lần.

Câu hỏi dẫn dắt: *có cách nào đếm tần suất của mọi từ chỉ bằng một lần duyệt không?* Có: dùng hash table.

## 6. Ý tưởng bước đột phá — đếm một lần, tra cứu O(1)

Insight: thay vì gọi `count` lặp lại, **duyệt mỗi mảng đúng một lần** để xây bảng tần suất. Sau đó mỗi câu hỏi "từ `w` xuất hiện mấy lần?" chỉ tốn `O(1)`.

Quy tắc quyết định (invariant của bảng đếm):

- **Nếu** `cnt1[w] == 1` **và** `cnt2[w] == 1` **thì** `w` được tính.
- **Ngược lại** (vắng mặt, xuất hiện ≥ 2 lần ở bất kỳ mảng nào) thì không.

Chỉ cần duyệt các khóa của một bảng (vì từ không có ở bảng kia sẽ có đếm `0`, tự loại).

## 7. Các cách tiếp theo

### Cách 2 — Hai `Counter`

Tạo `c1 = Counter(words1)`, `c2 = Counter(words2)`. Duyệt `w, k in c1.items()`: nếu `k == 1` và `c2[w] == 1` thì tăng đáp án. Nhờ `Counter`, `c2[w]` của từ vắng mặt là `0` nên không lỗi.

In [8]:
from collections import Counter

def count_words_two_counters(words1, words2):
    c1, c2 = Counter(words1), Counter(words2)
    return sum(1 for w, k in c1.items() if k == 1 and c2[w] == 1)

print(count_words_two_counters(words1, words2))

2


In [9]:
from typing import List
from collections import Counter

class SolutionTwoCounters:
    def countWords(self, words1: List[str], words2: List[str]) -> int:
        c1, c2 = Counter(words1), Counter(words2)
        return sum(1 for w, k in c1.items() if k == 1 and c2[w] == 1)

print(SolutionTwoCounters().countWords(["a", "ab"], ["a", "a", "a", "ab"]))   # 1

1


Độ phức tạp Cách 2: `O((n + m)·L)` thời gian (băm chuỗi dài `L`), `O(n + m)` bộ nhớ.

### Cách 3 — Hai tập "đúng 1 lần" rồi lấy giao

Dùng hai `set`: `seen` (đã gặp) và `dup` (đã gặp ≥ 2 lần). Với mỗi mảng, từ chỉ nằm trong `seen` mà không nằm trong `dup` là từ "đúng 1 lần". Cuối cùng `len(once1 & once2)`.

In [10]:
def once_set(words):
    seen, dup = set(), set()
    for w in words:
        if w in seen:
            dup.add(w)
        else:
            seen.add(w)
    return seen - dup

def count_words_sets(words1, words2):
    return len(once_set(words1) & once_set(words2))

print(once_set(words1), once_set(words2))
print(count_words_sets(words1, words2))

{'leetcode', 'as', 'amazing'} {'is', 'amazing', 'leetcode'}
2


In [11]:
from typing import List

class SolutionSets:
    def countWords(self, words1: List[str], words2: List[str]) -> int:
        def once(words):
            seen, dup = set(), set()
            for w in words:
                if w in seen:
                    dup.add(w)
                else:
                    seen.add(w)
            return seen - dup
        return len(once(words1) & once(words2))

print(SolutionSets().countWords(["b", "bb", "bbb"], ["a", "aa", "aaa"]))   # 0

0


Độ phức tạp Cách 3: `O((n + m)·L)` thời gian, `O(n + m)` bộ nhớ.

### Cách 4 — Một dictionary duy nhất với "điểm trạng thái"

Ý tưởng sáng tạo: dùng **một** `dict` `state[w]` chạy như máy trạng thái nhỏ.

- Duyệt `words1`: cộng `1` vào `state[w]` (nhưng không cho vượt quá 2 vì ≥ 2 đều là "hỏng").
- Duyệt `words2`: chỉ tăng nếu `state[w]` đang bằng `1` (đã có đúng 1 ở mảng 1) -> thành `11` theo nghĩa "đủ 2 mảng"; nếu `w` gặp lần nữa -> hỏng.

Để dễ nhớ ta dùng cặp đếm `[c1, c2]` trong một dict; cuối cùng đếm các từ có `[1, 1]`.

In [12]:
def count_words_pair_dict(words1, words2):
    pair = {}                      # w -> [số lần ở words1, số lần ở words2]
    for w in words1:
        pair.setdefault(w, [0, 0])[0] += 1
    for w in words2:
        pair.setdefault(w, [0, 0])[1] += 1
    return sum(1 for p in pair.values() if p == [1, 1])

print(count_words_pair_dict(words1, words2))
print(count_words_pair_dict(["a", "ab"], ["a", "a", "a", "ab"]))

2
1


In [13]:
from typing import List

class SolutionPairDict:
    def countWords(self, words1: List[str], words2: List[str]) -> int:
        pair = {}
        for w in words1:
            pair.setdefault(w, [0, 0])[0] += 1
        for w in words2:
            pair.setdefault(w, [0, 0])[1] += 1
        return sum(1 for p in pair.values() if p == [1, 1])

print(SolutionPairDict().countWords(["leetcode", "is", "amazing", "as", "is"], ["amazing", "leetcode", "is"]))

2


Độ phức tạp Cách 4: `O((n + m)·L)` thời gian, `O(n + m)` bộ nhớ. `setdefault(w, default)` trả giá trị hiện có hoặc tạo mới nếu chưa có.

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 1. Bảng đếm:

| Từ | `c1` | `c2` | Tính? |
|---|---|---|---|
| leetcode | 1 | 1 | có |
| is | 2 | 1 | không (2 lần ở words1) |
| amazing | 1 | 1 | có |
| as | 1 | 0 | không (vắng ở words2) |

Hàm dưới in từng bước duyệt.

In [14]:
def trace_two_counters(words1, words2):
    c1, c2 = Counter(words1), Counter(words2)
    print("c1 =", dict(c1))
    print("c2 =", dict(c2))
    ans = 0
    for w, k in c1.items():
        ok = (k == 1 and c2[w] == 1)
        print(f"w={w!r:10} c1={k} c2={c2[w]} -> {'TÍNH' if ok else 'bỏ'}")
        if ok:
            ans += 1
            print("   ans =", ans)
    return ans

print("Kết quả:", trace_two_counters(words1, words2))

c1 = {'leetcode': 1, 'is': 2, 'amazing': 1, 'as': 1}
c2 = {'amazing': 1, 'leetcode': 1, 'is': 1}
w='leetcode' c1=1 c2=1 -> TÍNH
   ans = 1
w='is'       c1=2 c2=1 -> bỏ
w='amazing'  c1=1 c2=1 -> TÍNH
   ans = 2
w='as'       c1=1 c2=0 -> bỏ
Kết quả: 2


In [15]:
print("Kết quả:", trace_two_counters(["a", "ab"], ["a", "a", "a", "ab"]))

c1 = {'a': 1, 'ab': 1}
c2 = {'a': 3, 'ab': 1}
w='a'        c1=1 c2=3 -> bỏ
w='ab'       c1=1 c2=1 -> TÍNH
   ans = 1
Kết quả: 1


## 9. Code nộp LeetCode

Dùng Cách 2 (hai `Counter`) — ngắn, rõ, nhanh.

In [16]:
from typing import List, Optional
from collections import Counter

class Solution:
    def countWords(self, words1: List[str], words2: List[str]) -> int:
        c1, c2 = Counter(words1), Counter(words2)
        return sum(1 for w, k in c1.items() if k == 1 and c2[w] == 1)

print(Solution().countWords(["leetcode", "is", "amazing", "as", "is"], ["amazing", "leetcode", "is"]))

2


## 10. Vì sao thuật toán đúng?

- `Counter` ghi đúng số lần xuất hiện của mỗi từ trong từng mảng (mỗi phần tử được cộng đúng một lần).
- Một từ thỏa đề **khi và chỉ khi** đếm ở `words1` bằng 1 và đếm ở `words2` bằng 1. Ta duyệt mọi từ của `words1` (đầy đủ khoảng tìm), nên không sót từ nào có đếm ở `words1` bằng 1.
- Từ chỉ có ở `words2` có `c1 = 0 != 1` nên không bao giờ thỏa; vì thế duyệt `c1` đã đủ.
- Mỗi từ khác nhau trong `c1` chỉ xuất hiện đúng một lần (khóa dict duy nhất) nên không đếm trùng.

Cách 3 đúng vì `seen - dup` giữ lại đúng các từ gặp 1 lần: lần gặp đầu vào `seen`, lần gặp thứ 2 trở đi vào `dup` và bị loại khi trừ.

In [17]:
# Kiểm chứng invariant của Cách 3: once_set(words) == {w : đếm == 1}
for ws in (words1, words2, ["a", "a", "a"], ["x"], []):
    expect = {w for w in set(ws) if ws.count(w) == 1}
    assert once_set(ws) == expect, ws
print("invariant OK")

invariant OK


## 11. Phân tích Time / Space Complexity

Gọi `n = len(words1)`, `m = len(words2)`, `L ≤ 30` là độ dài từ tối đa.

| Cách | Time | Space |
|---|---|---|
| 1. Brute `list.count` | `O(n·(n+m)·L)` | `O(n)` (set) |
| 2. Hai Counter | `O((n+m)·L)` | `O(n+m)` |
| 3. Hai set `seen/dup` | `O((n+m)·L)` | `O(n+m)` |
| 4. Dict cặp `[c1,c2]` | `O((n+m)·L)` | `O(n+m)` |

Băm một chuỗi dài `L` tốn `O(L)`; vì `L ≤ 30` coi như hằng số, thực tế tuyến tính `O(n+m)`.

## 12. Bộ test

Ví dụ của đề, edge case (một mảng toàn trùng, mảng một phần tử, không có từ chung) và đối chiếu ngẫu nhiên giữa **cả 4 cách**.

In [18]:
ALL = [count_words_brute, count_words_two_counters, count_words_sets, count_words_pair_dict]

cases = [
    (["leetcode", "is", "amazing", "as", "is"], ["amazing", "leetcode", "is"], 2),
    (["b", "bb", "bbb"], ["a", "aa", "aaa"], 0),
    (["a", "ab"], ["a", "a", "a", "ab"], 1),
    (["a"], ["a"], 1),
    (["a", "a"], ["a"], 0),
    (["a"], ["a", "a"], 0),
    (["a", "b", "c"], ["c", "b", "a"], 3),
    (["x", "y"], ["z"], 0),
]
for w1, w2, exp in cases:
    for f in ALL:
        assert f(w1, w2) == exp, (f.__name__, w1, w2, exp)
    assert Solution().countWords(w1, w2) == exp
print("Tất cả test của đề + edge case đều qua:", len(cases), "ca x", len(ALL) + 1, "cách")

Tất cả test của đề + edge case đều qua: 8 ca x 5 cách


In [19]:
import random
random.seed(2085)
vocab = ["a", "b", "c", "d", "e", "ab", "ba"]
for _ in range(3000):
    n, m = random.randint(1, 10), random.randint(1, 10)
    w1 = [random.choice(vocab) for _ in range(n)]
    w2 = [random.choice(vocab) for _ in range(m)]
    base = count_words_brute(w1, w2)
    assert count_words_two_counters(w1, w2) == base
    assert count_words_sets(w1, w2) == base
    assert count_words_pair_dict(w1, w2) == base
    assert Solution().countWords(w1, w2) == base
print("3000 test ngẫu nhiên: 4 cách + Solution khớp nhau")

3000 test ngẫu nhiên: 4 cách + Solution khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ kiểm tra "có mặt ở cả hai" mà quên điều kiện "đúng 1 lần"

Nhiều bạn đọc nhanh đề và đếm giao của hai tập từ.

In [20]:
def wrong_intersection(words1, words2):
    return len(set(words1) & set(words2))

print("sai:", wrong_intersection(words1, words2), "| đúng:", Solution().countWords(words1, words2))

sai: 3 | đúng: 2


Sai vì `"is"` có ở cả hai mảng nhưng xuất hiện 2 lần ở `words1`. Đề yêu cầu **đúng 1 lần ở mỗi mảng**. Cách sửa: đếm tần suất (Counter) rồi lọc `== 1`.

### Lỗi 2: Chỉ đếm `== 1` ở một mảng

In [21]:
def wrong_one_side(words1, words2):
    c1, c2 = Counter(words1), Counter(words2)
    return sum(1 for w, k in c1.items() if k == 1 and c2[w] >= 1)

t1, t2 = ["a"], ["a", "a", "a"]
print("sai:", wrong_one_side(t1, t2), "| đúng:", Solution().countWords(t1, t2))

sai: 1 | đúng: 0


`c2[w] >= 1` cho phép từ xuất hiện 3 lần ở `words2`. Phải là `c2[w] == 1`.

### Lỗi 3: Dùng `dict[w]` thường mà không xử lý khóa vắng mặt (`KeyError`)

In [22]:
def wrong_keyerror(words1, words2):
    c1, c2 = {}, {}
    for w in words1: c1[w] = c1.get(w, 0) + 1
    for w in words2: c2[w] = c2.get(w, 0) + 1
    return sum(1 for w, k in c1.items() if k == 1 and c2[w] == 1)   # c2[w] có thể KeyError

try:
    wrong_keyerror(["as"], ["is"])
except KeyError as e:
    print("KeyError:", e)

def fixed(words1, words2):
    c1, c2 = Counter(words1), Counter(words2)      # Counter trả 0 cho khóa vắng
    return sum(1 for w, k in c1.items() if k == 1 and c2[w] == 1)
print(fixed(["as"], ["is"]))

KeyError: 'as'
0


### Lỗi 4 (thêm): Đếm bằng `count` trong vòng lặp trên list gốc rồi cộng nhiều lần

Duyệt `for w in words1` (không dùng `set`) cũng ra đúng vì từ trùng có `count != 1`; nhưng nếu bạn đổi điều kiện sang `>= 1` thì từ trùng bị đếm nhiều lần. Hãy luôn nghĩ: **mỗi từ khác nhau chỉ nên được xét một lần**.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "xuất hiện đúng k lần", "tần suất", "trùng lặp" -> nghĩ ngay **Counter / hash map**.
- Thấy "ở cả hai mảng" -> nghĩ **giao tập hợp** hoặc tra cứu chéo `c2[w]`.
- `Counter[x]` cho `0` với khóa vắng; dùng để bỏ bớt `if w in c2`.
- Với 1000 phần tử, brute `O(n²)` vẫn qua nhưng hãy tập viết bản `O(n)`.
- Khi dữ liệu nhỏ (26 chữ cái, số 0–9), có thể dùng mảng đếm thay dict.

## 15. Biến thể của bài

1. Đếm từ xuất hiện đúng 1 lần ở **ít nhất một** mảng (hợp thay vì giao).
2. Đếm từ xuất hiện **đúng `k` lần** ở cả hai mảng (tổng quát hóa).
3. Trả về **danh sách** các từ chung thay vì số lượng, sắp xếp theo thứ tự chữ cái.
4. Với `k` mảng thay vì hai: đếm từ xuất hiện đúng 1 lần ở **mọi** mảng.
5. LeetCode 349 / 350 (Intersection of Two Arrays I/II) — cùng tư duy giao tập và đếm.

In [23]:
def count_exact_k(words1, words2, k):
    c1, c2 = Counter(words1), Counter(words2)
    return sum(1 for w, v in c1.items() if v == k and c2[w] == k)

print(count_exact_k(["a", "a", "b"], ["a", "a", "c"], 2))   # 1  ('a' xuất hiện đúng 2 lần ở cả hai)
print(count_exact_k(words1, words2, 1))                      # 2

1
2


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute `count` | Với mỗi từ, đếm lại bằng `list.count` | `O(n(n+m)L)` | `O(n)` | Học cơ bản, dữ liệu rất nhỏ |
| 2. Hai Counter | Đếm 2 mảng rồi lọc | `O((n+m)L)` | `O(n+m)` | **Nên dùng** khi thi, rõ ràng |
| 3. `seen/dup` set | Tách tập "đúng 1 lần", lấy giao | `O((n+m)L)` | `O(n+m)` | Khi muốn dùng phép tập hợp |
| 4. Dict cặp | Một dict giữ `[c1,c2]` | `O((n+m)L)` | `O(n+m)` | Khi mở rộng sang nhiều mảng |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 — Đoán output:** `Counter(["a","b","a"])["a"]` và `Counter(["a"])["z"]` bằng bao nhiêu?

**Bài 2 — Tính tay:** `words1 = ["x","y","y"]`, `words2 = ["x","y"]`. Đáp án của `countWords`?

**Bài 3 — Điền TODO:** hoàn thành `exercise_once(words)` trả về tập các từ xuất hiện đúng 1 lần.

**Bài 4 — Tự viết lại:** viết `exercise_count_words(words1, words2)` bằng một cách khác bạn thích.

**Bài 5 — Mini challenge:** viết `common_exactly_once_all(lists)` nhận danh sách các mảng từ, trả số từ xuất hiện đúng 1 lần ở **mọi** mảng.

Các hàm bên dưới chỉ định nghĩa (chưa gọi) để không gây lỗi khi chạy toàn bộ notebook.

In [24]:
def exercise_once(words):
    # TODO: trả về set các từ xuất hiện đúng 1 lần trong words
    pass

# print(exercise_once(["a", "b", "a"]))   # kỳ vọng {'b'}

In [25]:
def exercise_count_words(words1, words2):
    # TODO: tự viết bằng một cách khác (vd dict cặp, hoặc sort)
    pass

# assert exercise_count_words(["a", "ab"], ["a", "a", "a", "ab"]) == 1

In [26]:
def common_exactly_once_all(lists):
    # TODO: số từ xuất hiện đúng 1 lần ở MỌI mảng trong lists
    pass

# assert common_exactly_once_all([["a", "b"], ["a", "c"], ["a", "a"]]) == 0

## 18. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | `Counter` trả số lần xuất hiện | `"a"` xuất hiện 2 lần | Khóa vắng mặt cho `0` |
| 2 | Đếm từng mảng | `"y"` có 2 lần ở `words1` | Chỉ `"x"` đúng 1 lần ở cả hai |
| 3 | Dùng `Counter` | Lọc `k == 1` | `{w for w, k in Counter(words).items() if k == 1}` |
| 4 | Thử `seen/dup` | Hoặc dict cặp `[c1,c2]` | Chạy lại test mục 12 để đối chiếu |
| 5 | Dùng nhiều Counter | Duyệt từ của mảng đầu | Mọi mảng `Counter[w] == 1` -> `all(...)` |

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `2` và `0`. **Bài 2:** `1` (chỉ `"x"`).

In [27]:
assert Counter(["a", "b", "a"])["a"] == 2 and Counter(["a"])["z"] == 0
assert Solution().countWords(["x", "y", "y"], ["x", "y"]) == 1

def exercise_once_answer(words):
    return {w for w, k in Counter(words).items() if k == 1}

def exercise_count_words_answer(words1, words2):
    return count_words_pair_dict(words1, words2)

def common_exactly_once_all_answer(lists):
    counters = [Counter(l) for l in lists]
    return sum(1 for w in counters[0] if all(c[w] == 1 for c in counters))

assert exercise_once_answer(["a", "b", "a"]) == {"b"}
assert exercise_count_words_answer(["a", "ab"], ["a", "a", "a", "ab"]) == 1
assert common_exactly_once_all_answer([["a", "b"], ["a", "c"], ["a", "a"]]) == 0
assert common_exactly_once_all_answer([["a", "b"], ["a", "c"], ["a", "d", "d"]]) == 1
print("Đáp án bài tập: tất cả assert đều qua")

Đáp án bài tập: tất cả assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template**

```python
from collections import Counter
class Solution:
    def countWords(self, words1, words2):
        c1, c2 = Counter(words1), Counter(words2)
        return sum(1 for w, k in c1.items() if k == 1 and c2[w] == 1)
```

**Checklist**

- [ ] Đã đọc kỹ: "đúng 1 lần" ở **cả hai** mảng (không phải chỉ có mặt ở cả hai).
- [ ] Dùng `Counter` (khóa vắng mặt cho `0`), không bị `KeyError`.
- [ ] Mỗi từ khác nhau chỉ được xét một lần (duyệt `.items()`).
- [ ] Đã test ví dụ 1, 2, 3 và edge case (một phần tử, từ lặp nhiều lần).
- [ ] Biết độ phức tạp `O(n + m)` thời gian và `O(n + m)` bộ nhớ.